# Real-Time Hand Gesture Mouse Control

This notebook demonstrates how to control your system cursor using hand gestures captured from your webcam.

### Prerequisites
Install the required libraries before running the script:
```bash
pip install opencv-python mediapipe pyautogui pynput numpy
```

### Option 1: Run the `main.py` script directly from the notebook (Recommended)

Running the script via `%run` ensures that the OpenCV window runs in a separate execution flow and will not freeze your Jupyter notebook kernel when closed.

In [ ]:
%run main.py

### Option 2: Self-Contained Notebook Code

If you prefer running the code directly inside a notebook cell, run the cell below. 

*Note: Press **'q'** on your keyboard while focusing the OpenCV window to exit and release the webcam cleanly.*

In [ ]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import util
from pynput.mouse import Button, Controller

# Initialize controllers
mouse = Controller()
pyautogui.FAILSAFE = False
screen_width, screen_height = pyautogui.size()

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    max_num_hands=1
)

# Read camera frames
cap = cv2.VideoCapture(0)
print("Webcam started. Focus the pop-up window and press 'q' to quit.")

left_clicked = False
right_clicked = False
double_clicked = False
screenshot_taken = False

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)
        
        # Get landmark coordinates in pixels
        landmark_list = []
        if results.multi_hand_landmarks:
            for lm in results.multi_hand_landmarks[0].landmark:
                cx, cy = int(lm.x * w), int(lm.y * h)
                landmark_list.append((cx, cy))
        
        action_text = "Idle"
        
        if len(landmark_list) == 21:
            thumb_index_dist = util.get_distance([landmark_list[4], landmark_list[8]])
            thumb_pinky_dist = util.get_distance([landmark_list[4], landmark_list[20]])
            
            # Cursor movement
            index_tip = results.multi_hand_landmarks[0].landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
            x = np.interp(index_tip.x, [0.2, 0.8], [0, screen_width])
            y = np.interp(index_tip.y, [0.2, 0.8], [0, screen_height])
            pyautogui.moveTo(int(x), int(y))
            
            # Check gestures
            if (
                util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) < 50 and
                util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) < 50 and
                thumb_index_dist < 50
            ):
                action_text = "Double Click"
                if not double_clicked:
                    pyautogui.doubleClick()
                    double_clicked = True
            elif (
                util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) < 50 and
                util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) > 90 and
                thumb_index_dist < 50
            ):
                action_text = "Left Click"
                if not left_clicked:
                    pyautogui.click(button='left')
                    left_clicked = True
            elif (
                util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) < 50 and
                util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) > 90 and
                thumb_index_dist < 50
            ):
                action_text = "Right Click"
                if not right_clicked:
                    pyautogui.click(button='right')
                    right_clicked = True
            elif (
                util.get_angle(landmark_list[17], landmark_list[18], landmark_list[20]) < 50 and
                thumb_pinky_dist < 50
            ):
                action_text = "Screenshot"
                if not screenshot_taken:
                    pyautogui.screenshot("screenshot_notebook.png")
                    print("Screenshot saved!")
                    screenshot_taken = True
            else:
                left_clicked = False
                right_clicked = False
                double_clicked = False
                screenshot_taken = False
                action_text = "Moving Mouse"
            
            # Draw landmarks
            for lm in landmark_list:
                cv2.circle(frame, lm, 5, (0, 255, 0), -1)
        
        cv2.putText(frame, f"Action: {action_text}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        cv2.imshow("Hand Gesture Mouse Control (Jupyter)", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Camera released and windows closed.")